# Part 2 (notebook 1 of 2): Data preparation and baseline

This notebook prepares the data and measures the **base model** (Stable Diffusion 1.5 with no fine-tuning). The fine-tuning notebook (`p2_finetune.ipynb`) later reads the files saved here.

| Step | Output |
|---|---|
| 2.1 Data preparation | cleaned 512×512 train/test images, cleaning report |
| 20 fixed prompts | `eval/p2_prompts.json` (written before any training) |
| Baseline generation | 40 base-model images (20 prompts × seeds 0 and 1) |
| Baseline metrics | KID, CLIP score, LPIPS diversity, NSFW flags → `eval/p2_baseline_results.csv` |

**Base model:** `stable-diffusion-v1-5/stable-diffusion-v1-5`. It is a *latent* diffusion model (Rombach et al., 2022). A VAE compresses each 512×512 image to a 64×64×4 latent, a CLIP text encoder reads the prompt, and a UNet removes noise step by step, guided by the text.

In [ ]:
%pip install -q diffusers==0.31.0 transformers==4.46.3 peft==0.13.2 accelerate==1.1.1 datasets==3.1.0 huggingface_hub==0.26.2 torchmetrics==1.6.0 torch-fidelity==0.3.0 lpips==0.1.4 imagehash==4.3.1

In [ ]:
import subprocess, platform, random, os
import numpy as np, torch, diffusers, transformers, peft, torchmetrics, datasets

SEED = 266
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

try:
    commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
except Exception:
    commit = "no git repo here"

print("commit      ", commit)
print("python      ", platform.python_version())
for m in [torch, diffusers, transformers, peft, torchmetrics, datasets]:
    print(f"{m.__name__:12s}", m.__version__)
print("gpu         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("seed        ", SEED)

In [ ]:
import json, time, gc, hashlib, io, textwrap
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import imagehash
import torch.nn.functional as F

In [ ]:
from datasets import load_dataset, Image as HFImage
from huggingface_hub import dataset_info
from transformers import CLIPImageProcessor
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from diffusers.pipelines.stable_diffusion.safety_checker import StableDiffusionSafetyChecker
from torchmetrics.image.kid import KernelInceptionDistance
from torchmetrics.multimodal.clip_score import CLIPScore
import lpips

In [ ]:
device = "cuda"
BASE = "stable-diffusion-v1-5/stable-diffusion-v1-5"
DATASET = "lambdalabs/naruto-blip-captions"
RES = 512

In [ ]:
IN_COLAB = "google.colab" in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/data266-lab2")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EVAL = ROOT / "eval"
WORK = ROOT / "work"
DATA = WORK / "p2_data"
GEN_DIR = WORK / "p2_gen"
LOGS = ROOT / "logs"
for d in [EVAL, DATA, GEN_DIR, LOGS]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
LOG_FILE = LOGS / "p2_run_log.txt"

def log(*args, file=LOG_FILE):
    line = time.strftime("%Y-%m-%d %H:%M:%S") + " | " + " ".join(str(a) for a in args)
    print(line)
    with open(file, "a") as f:
        f.write(line + "\n")

log("=" * 60)
log("baseline notebook | seed", SEED, "| gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
log("commit", commit, "| torch", torch.__version__, "| diffusers", diffusers.__version__)

## 2.1 Data preparation

Steps: load the dataset → remove broken images and duplicates → split 90/10 with seed 266 → crop and resize to 512×512 → NSFW filter on training images → describe the data.

- **Corrupted:** the file can't be opened, is tiny (<64 px) or has an empty caption.
- **Exact duplicate:** same file bytes (MD5 hash).
- **Near duplicate:** perceptual hash differs by ≤ 2 bits, i.e. the same picture saved slightly differently.

In [ ]:
raw = load_dataset(DATASET, split="train").cast_column("image", HFImage(decode=False))
raw

In [ ]:
info = dataset_info(DATASET)
DATA_LICENSE = (info.card_data or {}).get("license")
if DATA_LICENSE is None:
    DATA_LICENSE = "not stated on the dataset card; images from narutopedia.com (Naruto is copyrighted), non-commercial research use only"
print("dataset license:", DATA_LICENSE)

In [ ]:
def img_bytes(x):
    if x.get("bytes"):
        return x["bytes"]
    return open(x["path"], "rb").read()

In [ ]:
keep, corrupted, exact_dup, near_dup = [], [], [], []
seen_md5 = set()
seen_phash = []

for i, row in enumerate(raw):
    try:
        b = img_bytes(row["image"])
        Image.open(io.BytesIO(b)).verify()
        im = Image.open(io.BytesIO(b)).convert("RGB")
        if min(im.size) < 64 or not str(row["text"]).strip():
            raise ValueError
    except Exception:
        corrupted.append(i)
        continue

    md5 = hashlib.md5(b).hexdigest()
    if md5 in seen_md5:
        exact_dup.append(i)
        continue

    ph = imagehash.phash(im)
    if any(ph - h <= 2 for h in seen_phash):
        near_dup.append(i)
        continue

    seen_md5.add(md5)
    seen_phash.append(ph)
    keep.append(i)

In [ ]:
pd.Series({
    "original": len(raw),
    "corrupted / empty caption": len(corrupted),
    "exact duplicates": len(exact_dup),
    "near duplicates (phash<=2)": len(near_dup),
    "removed total": len(raw) - len(keep),
    "kept": len(keep),
}, name="count")

In [ ]:
cleaning = {
    "dataset": DATASET,
    "license": DATA_LICENSE,
    "original": len(raw),
    "corrupted": len(corrupted),
    "exact_duplicates": len(exact_dup),
    "near_duplicates": len(near_dup),
    "kept": len(keep),
    "removed_ids": {"corrupted": corrupted, "exact_dup": exact_dup, "near_dup": near_dup},
}

In [ ]:
split = raw.select(keep).train_test_split(test_size=0.1, seed=SEED)
train_raw = split["train"]
test_raw = split["test"]
log("cleaning | original", len(raw), "| corrupted", len(corrupted), "| exact dup", len(exact_dup),
    "| near dup", len(near_dup), "| kept", len(keep))
log("split | train", len(train_raw), "| test", len(test_raw))

In [ ]:
def to_square(b, size=RES):
    im = Image.open(io.BytesIO(b)).convert("RGB")
    w, h = im.size
    s = min(w, h)
    im = im.crop(((w - s) // 2, (h - s) // 2, (w + s) // 2, (h + s) // 2))
    return im.resize((size, size), Image.BICUBIC)

In [ ]:
def dump(ds, name):
    out = DATA / name
    out.mkdir(exist_ok=True)
    meta = []
    for j, row in enumerate(ds):
        p = out / f"{j:04d}.png"
        if not p.exists():
            to_square(img_bytes(row["image"])).save(p)
        meta.append({"file": str(p), "text": row["text"]})
    df = pd.DataFrame(meta)
    df.to_csv(DATA / f"{name}.csv", index=False)
    return df

In [ ]:
train_df = dump(train_raw, "train")
test_df = dump(test_raw, "test")
print(Image.open(train_df.file[0]).size, len(train_df), len(test_df))

### NSFW filter on training images

We use the official Stable Diffusion safety checker (`CompVis/stable-diffusion-safety-checker`), a CLIP-based classifier. Flagged training images are removed. It is known to be conservative on drawn characters, so some flags are probably false positives.

In [ ]:
safety = StableDiffusionSafetyChecker.from_pretrained("CompVis/stable-diffusion-safety-checker").to(device)
safety_proc = CLIPImageProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
def nsfw_flags(paths, bs=32):
    flags = []
    for k in range(0, len(paths), bs):
        ims = [Image.open(p).convert("RGB") for p in paths[k:k + bs]]
        clip_in = safety_proc(ims, return_tensors="pt").pixel_values.to(device)
        arr = np.stack([np.asarray(im, dtype=np.float32) / 255 for im in ims])
        _, has = safety(images=arr, clip_input=clip_in)
        flags += [bool(h) for h in has]
    return flags

In [ ]:
train_df["nsfw"] = nsfw_flags(train_df.file.tolist())
n_train_nsfw = int(train_df.nsfw.sum())
print("training images flagged NSFW:", n_train_nsfw)

In [ ]:
train_df = train_df[~train_df.nsfw].reset_index(drop=True)
train_df.to_csv(DATA / "train_clean.csv", index=False)
log("nsfw filter on train | flagged", n_train_nsfw, f"({n_train_nsfw / (len(train_df) + n_train_nsfw):.1%})",
    "| train left", len(train_df))

In [ ]:
cleaning.update({"train": len(train_df), "test": len(test_df), "train_nsfw_removed": n_train_nsfw,
                 "split_seed": SEED, "resolution": RES})
(EVAL / "p2_data_cleaning.json").write_text(json.dumps(cleaning, indent=2))
{k: v for k, v in cleaning.items() if k != "removed_ids"}

### Dataset description

In [ ]:
words = train_df.text.str.split().str.len()
print("caption length (words): mean %.1f, min %d, max %d" % (words.mean(), words.min(), words.max()))

In [ ]:
train_df.text.str.split().str[:2].str.join(" ").value_counts().head(8)

In [ ]:
{k: round(train_df.text.str.contains(rf"\b{k}\b").mean(), 3) for k in ["man", "woman", "boy", "girl"]}

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (_, r) in zip(axes.flat, train_df.sample(8, random_state=SEED).iterrows()):
    ax.imshow(Image.open(r.file))
    ax.axis("off")
    ax.set_title(textwrap.fill(r.text, 30), fontsize=8)
plt.tight_layout()
plt.show()

**Dataset.** Naruto BLIP Captions: images of characters from the Naruto anime, collected from narutopedia.com, each with a short caption generated automatically by BLIP. The cleaned train/test sizes are printed above. Every image is center-cropped and resized to 512×512.

**Limitations / biases.**
1. Captions are machine-generated, short (about 9 words) and generic. They describe pose and colors but almost never name the character, and some are wrong, so the model learns the art style more than identities.
2. Gender and framing are unbalanced: about half the captions say "a man", far fewer say "woman" or "girl", and most images are single-character close-ups. Multi-character scenes and backgrounds are rare, which matters for the compositional prompts.

## Fixed evaluation prompts (written before training, saved to `eval/p2_prompts.json`)

- **10 in-domain:** real captions from the *test* set, so the model has never seen these exact image–caption pairs.
- **5 out-of-domain:** things that are not in Naruto (cars, food, dogs). These test whether fine-tuning broke general ability.
- **5 compositional:** several objects with colors and positions. These test attribute binding, which SD 1.5 is weak at.

If the file already exists, it is reused, so the prompts never change after the first run.

In [ ]:
in_domain = test_df.sample(10, random_state=SEED).text.tolist()

ood = [
    "a red sports car parked on a rainy city street",
    "a lighthouse on a rocky cliff at sunset",
    "a bowl of fruit on a wooden kitchen table",
    "an astronaut floating above the earth",
    "a golden retriever running on a beach",
]

comp = [
    "two characters, one in red and one in blue",
    "a girl with pink hair standing to the left of a boy with yellow hair",
    "three ninjas standing in a row, the middle one wearing a mask",
    "a man holding a sword in his left hand and a scroll in his right hand",
    "a small orange fox sitting on top of a large green frog",
]

In [ ]:
prompt_file = EVAL / "p2_prompts.json"
if prompt_file.exists():
    prompts = json.loads(prompt_file.read_text())
    print("using saved prompts from", prompt_file)
else:
    items = [("in_domain", p) for p in in_domain] + [("ood", p) for p in ood] + [("compositional", p) for p in comp]
    prompts = [{"id": i, "category": c, "prompt": p} for i, (c, p) in enumerate(items)]
    prompt_file.write_text(json.dumps(prompts, indent=2))
    log("prompts saved before training |", len(prompts), "prompts | md5", hashlib.md5(prompt_file.read_bytes()).hexdigest())
pd.DataFrame(prompts)

In [ ]:
counts = pd.Series([p["category"] for p in prompts]).value_counts()
assert len(prompts) == 20
assert counts["in_domain"] == 10 and counts["ood"] == 5 and counts["compositional"] == 5
counts

## Baseline generation (base model, no fine-tuning)

Sampler settings are fixed here and must be the same in the fine-tuning notebook: DPM-Solver++, 30 steps, CFG 7.5, 512×512, seeds 0 and 1. That gives 20 prompts × 2 seeds = 40 images.

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained(BASE, torch_dtype=torch.float16, safety_checker=None,
                                               requires_safety_checker=False).to(device)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.set_progress_bar_config(disable=True)

In [ ]:
GEN = dict(num_inference_steps=30, guidance_scale=7.5, height=RES, width=RES)
(EVAL / "p2_sampler.json").write_text(json.dumps({"scheduler": type(pipe.scheduler).__name__, **GEN}, indent=2))
print(type(pipe.scheduler).__name__, GEN)

In [ ]:
def generate(model):
    rows = []
    for p in prompts:
        for seed in (0, 1):
            path = GEN_DIR / f"{model}_p{p['id']:02d}_s{seed}.png"
            t0 = time.time()
            img = pipe(p["prompt"], generator=torch.Generator(device).manual_seed(seed), **GEN).images[0]
            sec = time.time() - t0
            img.save(path)
            rows.append({"model": model, "prompt_id": p["id"], "category": p["category"],
                         "prompt": p["prompt"], "seed": seed, "file": str(path), "sec": sec})
    return rows

In [ ]:
base = pd.DataFrame(generate("base"))
log("baseline generation |", len(base), "images | median sec/img", round(base.sec.median(), 2))
len(base)

In [ ]:
def show_row(ids, seed=0, title=None):
    fig, axes = plt.subplots(1, len(ids), figsize=(3.2 * len(ids), 3.8))
    for ax, pid in zip(axes, ids):
        f = base[(base.prompt_id == pid) & (base.seed == seed)].file.iloc[0]
        ax.imshow(Image.open(f))
        ax.axis("off")
        ax.set_title(textwrap.fill(f"[{pid}] {prompts[pid]['prompt']}", 26), fontsize=8)
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
show_row([p["id"] for p in prompts if p["category"] == "in_domain"][:5], title="base model, in-domain (seed 0)")

In [ ]:
show_row([p["id"] for p in prompts if p["category"] == "ood"], title="base model, out-of-domain (seed 0)")

In [ ]:
show_row([p["id"] for p in prompts if p["category"] == "compositional"], title="base model, compositional (seed 0)")

## Baseline metrics

| Metric | What it measures | Better |
|---|---|---|
| KID | distance between Inception features of the 20 in-domain generations and the 123 test images | lower |
| CLIP score | how well each image matches its prompt (100 × cosine similarity) | higher |
| LPIPS | perceptual distance between seed 0 and seed 1 of the same prompt (diversity) | higher = more varied |

KID is used instead of FID because it is unbiased with small samples. With only 20 images it is still noisy, so we also report the standard deviation over 100 subsets.

In [ ]:
def load_uint8(files, size=None):
    ims = [Image.open(f).convert("RGB") for f in files]
    if size:
        ims = [im.resize((size, size), Image.BICUBIC) for im in ims]
    return torch.from_numpy(np.stack([np.asarray(im) for im in ims])).permute(0, 3, 1, 2)

In [ ]:
real = load_uint8(test_df.file, 299)
fake = load_uint8(base[base.category == "in_domain"].file, 299)
real.shape, fake.shape

In [ ]:
kid = KernelInceptionDistance(subset_size=min(len(fake), len(real)), subsets=100).to(device)
torch.manual_seed(SEED)
for k in range(0, len(real), 32):
    kid.update(real[k:k + 32].to(device), real=True)
kid.update(fake.to(device), real=False)
kid_mu, kid_sd = [v.item() for v in kid.compute()]
log(f"baseline KID = {kid_mu:.4f} ± {kid_sd:.4f} (fake={len(fake)}, real={len(real)})")

In [ ]:
clip_metric = CLIPScore(model_name_or_path="openai/clip-vit-base-patch16").to(device)

scores = []
for r in base.itertuples():
    scores.append(clip_metric(load_uint8([r.file])[0].to(device), r.prompt).item())
    clip_metric.reset()
base["clip"] = scores
base.groupby("category").clip.mean().round(2)

In [ ]:
lp = lpips.LPIPS(net="alex").to(device)

def to_lp(f):
    x = np.asarray(Image.open(f).convert("RGB"), dtype=np.float32) / 127.5 - 1
    return torch.from_numpy(x).permute(2, 0, 1)[None].to(device)

In [ ]:
div = []
for pid, g in base.groupby("prompt_id"):
    a, b = g.sort_values("seed").file.tolist()
    with torch.no_grad():
        d = lp(to_lp(a), to_lp(b)).item()
    div.append({"model": "base", "prompt_id": pid, "category": g.category.iloc[0], "lpips": d})
div = pd.DataFrame(div)
div.groupby("category").lpips.mean().round(3)

In [ ]:
base["nsfw"] = nsfw_flags(base.file.tolist())
print("base generations flagged NSFW:", int(base.nsfw.sum()), "/", len(base))

## Baseline results table

In [ ]:
baseline = pd.Series({
    "KID in-domain (lower better)": kid_mu,
    "KID std": kid_sd,
    "CLIP all": base.clip.mean(),
    "CLIP in_domain": base[base.category == "in_domain"].clip.mean(),
    "CLIP ood": base[base.category == "ood"].clip.mean(),
    "CLIP compositional": base[base.category == "compositional"].clip.mean(),
    "LPIPS diversity": div.lpips.mean(),
    "NSFW flagged": int(base.nsfw.sum()),
    "median sec/img (gen)": base.sec.median(),
}, name="base").round(4)
baseline

In [ ]:
baseline.to_frame().to_csv(EVAL / "p2_baseline_results.csv")
base.to_csv(GEN_DIR / "base_metadata.csv", index=False)
div.to_csv(GEN_DIR / "base_lpips.csv", index=False)
log("baseline results\n" + baseline.to_string())

## Saved for the fine-tuning notebook

In [ ]:
for f in [DATA / "train_clean.csv", DATA / "test.csv", EVAL / "p2_prompts.json", EVAL / "p2_sampler.json",
          EVAL / "p2_data_cleaning.json", EVAL / "p2_baseline_results.csv", GEN_DIR / "base_metadata.csv",
          GEN_DIR / "base_lpips.csv"]:
    print(f"{'ok ' if f.exists() else 'MISSING'}  {f}")